# NFL Pillar Scoring v3 — Stitching Ablation Study

**Research question**: Does phrase stitching help or hurt pillar scoring?

**Hypothesis against stitching**:
- TF-IDF with `ngram_range=(1,2)` already handles `"pass rush"` as a native bigram
- Stitched tokens have lower document frequency → higher IDF → potentially over-weighted
- W2V trains on fewer effective tokens, reducing neighbor quality

**Hypothesis for stitching**:
- W2V needs single tokens — without stitching, compound meaning is lost in W2V space
- Custom stopword removal can accidentally split compound concepts (e.g., `game-ready` → `game` removed by stopwords)

We run both pipelines end-to-end and compare on:
1. W2V neighbor quality (semantic coherence)
2. Archetype vector sparsity (non-zero features)
3. Score distributions and variance
4. Rank correlation between the two versions (Spearman ρ)
5. Top-10 player agreement per pillar
6. Positional profile separation

In [1]:
import re
import warnings
import numpy as np
import pandas as pd
from collections import Counter
from scipy.stats import spearmanr

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.collocations import BigramCollocationFinder
from nltk.metrics import BigramAssocMeasures

from gensim.models import Word2Vec

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

import scipy.sparse as sp

for resource in ['punkt', 'stopwords', 'wordnet', 'omw-1.4', 'punkt_tab']:
    nltk.download(resource, quiet=True)

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.float_format', '{:.3f}'.format)

# ── Controls (shared) ─────────────────────────────────────────────────────────
PMI_TOP_N    = 30
PMI_MIN_FREQ = 5
TOP_TERMS_N  = 5
SEED_MODE    = '2-bin'

W_STRENGTHS  =  1.0
W_WEAKNESSES = -1.0
W_OVERVIEW   =  0.0

W2V_DIM       = 100
W2V_WINDOW    = 6
W2V_MIN_COUNT = 3
W2V_EPOCHS    = 30
W2V_SG        = 1
SEED_TOPN     = 20
SIM_THRESHOLD = 0.35
MIN_SEED_COUNT = 2
MAX_W2V_TERMS  = 25

In [2]:
df = pd.read_csv('../data/processed/draft_enriched_with_contracts.csv')

keep_cols = ['player_name', 'Pos_Group', 'position', 'grade', 'year',
             'made_it_contract', 'overview', 'strengths', 'weaknesses']
df = df[keep_cols].copy()

for col in ['overview', 'strengths', 'weaknesses']:
    df[col] = df[col].fillna('')

df = df[df[['overview', 'strengths', 'weaknesses']].apply(
    lambda r: any(r.str.strip() != ''), axis=1
)].reset_index(drop=True)

print(f'Players: {len(df)}')

Players: 6688


## 1. Shared vocabulary: stopwords, lemmatizer

In [3]:
KEEP_WORDS = {
    'high', 'low', 'heavy', 'light', 'deep', 'short', 'long', 'wide',
    'hard', 'soft', 'strong', 'quick', 'good', 'great', 'up', 'down',
    'off', 'out', 'over', 'through', 'above', 'below',
}

CUSTOM_STOPS = {
    'prospect', 'player', 'players', 'show', 'shows', 'need', 'needs',
    'ability', 'also', 'often', 'must', 'well', 'still', 'use', 'get',
    'make', 'look', 'help', 'time', 'year', 'team', 'game',
    'continue', 'develop', 'development', 'nfl', 'draft', 'college',
    'level', 'type', 'project', 'potential', 'upside', 'ceiling',
}

_base = set(stopwords.words('english'))
NFL_STOPWORDS = (_base - KEEP_WORDS) | CUSTOM_STOPS

lemmatizer = WordNetLemmatizer()

print(f'Stop list size: {len(NFL_STOPWORDS)}')

Stop list size: 224


## 2. Pipeline A — WITH stitching (v3 baseline)

In [4]:
_CURATED_RAW = {
    'change of direction':  'change_of_direction',
    'low pad level':        'low_pad_level',
    'run after catch':      'run_after_catch',
    'yards after contact':  'yards_after_contact',
    'yards after catch':    'yards_after_catch',
    'off the line':         'off_the_line',
    'off the ball':         'off_the_ball',
    'point of attack':      'point_of_attack',
    'pass rush':            'pass_rush',
    'pass rusher':          'pass_rusher',
    'pass protection':      'pass_protection',
    'pass coverage':        'pass_coverage',
    'pad level':            'pad_level',
    'press coverage':       'press_coverage',
    'man coverage':         'man_coverage',
    'zone coverage':        'zone_coverage',
    'ball skills':          'ball_skills',
    'ball hawk':            'ball_hawk',
    'ball carrier':         'ball_carrier',
    'body control':         'body_control',
    'contact balance':      'contact_balance',
    'closing speed':        'closing_speed',
    'lateral quickness':    'lateral_quickness',
    'quick twitch':         'quick_twitch',
    'high motor':           'high_motor',
    'first step':           'first_step',
    'get off':              'get_off',
    'hand fighting':        'hand_fighting',
    'hand strength':        'hand_strength',
    'block shedding':       'block_shedding',
    'anchor strength':      'anchor_strength',
    'route running':        'route_running',
    'run blocking':         'run_blocking',
    'open field':           'open_field',
    'red zone':             'red_zone',
    'second level':         'second_level',
    'hip flexibility':      'hip_flexibility',
    'soft hands':           'soft_hands',
    'heavy hands':          'heavy_hands',
    'strong hands':         'strong_hands',
    'short area':           'short_area',
    'three down':           'three_down',
    'top end':              'top_end',
    'two gap':              'two_gap',
    'one gap':              'one_gap',
    'snap count':           'snap_count',
    'game ready':           'game_ready',
    'hard worker':          'hard_worker',
    'work ethic':           'work_ethic',
}

CURATED_PHRASE_MAP = dict(
    sorted(_CURATED_RAW.items(), key=lambda x: len(x[0]), reverse=True)
)
print(f'Curated phrases: {len(CURATED_PHRASE_MAP)}')

Curated phrases: 49


In [5]:
def preprocess_stitch(text: str, phrase_map: dict = CURATED_PHRASE_MAP,
                      extra_phrases: dict = None) -> str:
    """WITH stitching (v3 baseline)."""
    if not isinstance(text, str) or not text.strip():
        return ''
    text = text.lower()
    text = re.sub(r'[-\u2013\u2014]', ' ', text)
    for phrase, token in phrase_map.items():
        text = text.replace(phrase, token)
    if extra_phrases:
        for phrase, token in extra_phrases.items():
            text = text.replace(phrase, token)
    text = re.sub(r'[^a-z_\s]', ' ', text)
    tokens = text.split()
    tokens = [t for t in tokens if '_' in t or t not in NFL_STOPWORDS]
    tokens = [t if '_' in t else lemmatizer.lemmatize(t) for t in tokens]
    tokens = [t for t in tokens if len(t) > 1]
    return ' '.join(tokens)


# Initial pass for PMI discovery
df['str_v1_stitch'] = df['strengths'].apply(preprocess_stitch)

# PMI bigram discovery
_token_lists = [
    [t for t in text.split() if '_' not in t]
    for text in df['str_v1_stitch']
]
finder = BigramCollocationFinder.from_documents(_token_lists)
finder.apply_freq_filter(PMI_MIN_FREQ)
scored = finder.score_ngrams(BigramAssocMeasures.pmi)

_already = set(CURATED_PHRASE_MAP.keys())
auto_candidates = [
    (w1, w2, round(score, 3), finder.ngram_fd[(w1, w2)])
    for (w1, w2), score in scored
    if  w1 not in NFL_STOPWORDS and w2 not in NFL_STOPWORDS
    and w1.isalpha() and w2.isalpha()
    and len(w1) > 2 and len(w2) > 2
    and f'{w1} {w2}' not in _already
][:PMI_TOP_N]

pmi_df = pd.DataFrame(auto_candidates, columns=['word1', 'word2', 'pmi', 'freq'])
pmi_df['phrase'] = pmi_df['word1'] + ' ' + pmi_df['word2']
pmi_df['token']  = pmi_df['word1'] + '_' + pmi_df['word2']
AUTO_PHRASE_MAP = dict(sorted(
    dict(zip(pmi_df['phrase'], pmi_df['token'])).items(),
    key=lambda x: len(x[0]), reverse=True
))

print(f'Auto-discovered bigrams: {len(AUTO_PHRASE_MAP)}')
print(pmi_df[['phrase', 'token', 'pmi', 'freq']].to_string(index=False))

# Final preprocessing WITH stitching
for raw, clean in [('strengths', 'str_s'), ('overview', 'ov_s'), ('weaknesses', 'wk_s')]:
    df[clean] = df[raw].apply(lambda t: preprocess_stitch(t, extra_phrases=AUTO_PHRASE_MAP))

df.drop(columns=['str_v1_stitch'], inplace=True)

vocab_s = set(t for col in ['str_s', 'ov_s', 'wk_s'] for text in df[col] for t in text.split())
stitched = [t for t in vocab_s if '_' in t]
print(f'\nVocab WITH stitching : {len(vocab_s):,} tokens ({len(stitched)} stitched)')

Auto-discovered bigrams: 30
               phrase                 token    pmi  freq
        freight train         freight_train 15.065     8
             rag doll              rag_doll 15.065     9
          blue collar           blue_collar 14.913     7
         calling card          calling_card 14.913     7
        signal caller         signal_caller 14.913     6
       deeper deepest        deeper_deepest 14.702     7
          wake forest           wake_forest 14.691     6
           notre dame            notre_dame 14.650    12
        battering ram         battering_ram 14.623     9
       internal clock        internal_clock 14.360     6
      seeking missile       seeking_missile 13.972    10
           grows root            grows_root 13.872     7
             peek boo              peek_boo 13.843     6
          phone booth           phone_booth 13.775    20
           stat sheet            stat_sheet 13.724    15
          world class           world_class 13.724     5
   

## 3. Pipeline B — WITHOUT stitching

Key differences:
- No phrase replacement (curated or PMI)
- No underscores → `token_pattern=r'[a-z]+'` in TF-IDF
- TF-IDF `ngram_range=(1,2)` naturally captures `"pass rush"` as a bigram
- W2V trains on individual words; compound meaning relies on proximity context
- Anchor seeds translated to unstitched equivalents

In [6]:
def preprocess_no_stitch(text: str) -> str:
    """WITHOUT stitching — pure token-level preprocessing."""
    if not isinstance(text, str) or not text.strip():
        return ''
    text = text.lower()
    text = re.sub(r'[-\u2013\u2014]', ' ', text)   # normalize hyphens
    text = re.sub(r'[^a-z\s]', ' ', text)           # no underscore preservation
    tokens = text.split()
    tokens = [t for t in tokens if t not in NFL_STOPWORDS]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    tokens = [t for t in tokens if len(t) > 1]
    return ' '.join(tokens)


for raw, clean in [('strengths', 'str_n'), ('overview', 'ov_n'), ('weaknesses', 'wk_n')]:
    df[clean] = df[raw].apply(preprocess_no_stitch)

vocab_n = set(t for col in ['str_n', 'ov_n', 'wk_n'] for text in df[col] for t in text.split())
print(f'Vocab WITHOUT stitching: {len(vocab_n):,} tokens (0 stitched)')

# Show what happens to compound terms
print('\nSample — same player, both versions (strengths):')
for _, row in df.head(2).iterrows():
    print(f"  [{row['position']}] {row['player_name']}")
    print(f"    RAW     : {row['strengths'][:100]}")
    print(f"    STITCH  : {row['str_s'][:100]}")
    print(f"    NO-STITCH: {row['str_n'][:100]}")
    print()

Vocab WITHOUT stitching: 13,944 tokens (0 stitched)

Sample — same player, both versions (strengths):
  [WR] Seyi Ajirotutu
    RAW     : Ajirotutu has ideal height and a muscular body.  Long-strider with very good straight-line speed.  F
    STITCH  : ajirotutu ideal height muscular body long strider good straight line speed fluid athlete sink hip ga
    NO-STITCH: ajirotutu ideal height muscular body long strider good straight line speed fluid athlete sink hip ga

  [DE] Rahim Alem
    RAW     : Alem is a high-motor kid who plays to the whistle every snap.  He can be a disruptive pass rusher wh
    STITCH  : alem high_motor kid play whistle every snap disruptive pass_rusher shed blocker good hand technique 
    NO-STITCH: alem high motor kid play whistle every snap disruptive pas rusher shed blocker good hand technique n



## 4. Train Word2Vec — both corpora

In [7]:
def train_w2v(clean_cols, label):
    all_clean = pd.concat([df[c] for c in clean_cols], ignore_index=True)
    sentences = [text.split() for text in all_clean if text.strip()]
    model = Word2Vec(
        sentences,
        vector_size = W2V_DIM,
        window      = W2V_WINDOW,
        min_count   = W2V_MIN_COUNT,
        epochs      = W2V_EPOCHS,
        sg          = W2V_SG,
        seed        = 42,
        workers     = 4,
    )
    print(f'W2V [{label}]: {len(model.wv):,} tokens in vocabulary')
    return model


w2v_s = train_w2v(['str_s', 'ov_s', 'wk_s'], 'WITH stitch')
w2v_n = train_w2v(['str_n', 'ov_n', 'wk_n'], 'NO stitch')

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

W2V [WITH stitch]: 7,845 tokens in vocabulary


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

W2V [NO stitch]: 7,758 tokens in vocabulary


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [8]:
# ── W2V quality comparison: neighbor coherence for key concepts ───────────────
# We probe both models with shared unstitched anchor words
PROBE_TERMS = ['explosive', 'technique', 'instinct', 'speed', 'leverage',
               'motor', 'agility', 'toughness', 'footwork', 'burst']

print('W2V NEIGHBOR QUALITY COMPARISON')
print('=' * 80)
for term in PROBE_TERMS:
    in_s = term in w2v_s.wv
    in_n = term in w2v_n.wv
    if not in_s and not in_n:
        continue
    print(f'\n{term.upper()}')
    if in_s:
        nbrs_s = [(w, round(sim, 3)) for w, sim in w2v_s.wv.most_similar(term, topn=8)]
        print(f'  WITH stitch : {nbrs_s}')
    if in_n:
        nbrs_n = [(w, round(sim, 3)) for w, sim in w2v_n.wv.most_similar(term, topn=8)]
        print(f'  NO stitch   : {nbrs_n}')

# Also check stitched-specific terms in stitch model
print('\n\nSTITCHED TOKEN NEIGHBORS (stitch model only):')
for tok in ['pass_rush', 'high_motor', 'quick_twitch', 'route_running', 'pad_level']:
    if tok in w2v_s.wv:
        nbrs = [(w, round(sim, 3)) for w, sim in w2v_s.wv.most_similar(tok, topn=6)]
        print(f'  {tok}: {nbrs}')
    else:
        print(f'  {tok}: NOT IN VOCAB')

W2V NEIGHBOR QUALITY COMPARISON

EXPLOSIVE
  WITH stitch : [('explosion', 0.627), ('explosiveness', 0.567), ('silatolu', 0.557), ('quick_twitch', 0.548), ('casting', 0.546), ('elite', 0.542), ('whittaker', 0.535), ('redmond', 0.52)]
  NO stitch   : [('explosion', 0.633), ('explosiveness', 0.592), ('silatolu', 0.575), ('twitched', 0.556), ('blazing', 0.525), ('redmond', 0.521), ('presser', 0.521), ('elite', 0.519)]

TECHNIQUE
  WITH stitch : [('coached', 0.584), ('fundamental', 0.567), ('consistency', 0.53), ('usage', 0.524), ('byers', 0.522), ('kilgore', 0.514), ('teamwork', 0.514), ('chewing', 0.512)]
  NO stitch   : [('fundamental', 0.54), ('consistency', 0.519), ('byers', 0.517), ('usage', 0.509), ('teamwork', 0.497), ('gilmore', 0.492), ('weaponize', 0.49), ('kilgore', 0.489)]

INSTINCT
  WITH stitch : [('awareness', 0.698), ('recognition', 0.663), ('verner', 0.618), ('ball_skills', 0.585), ('stuckey', 0.58), ('pinkston', 0.572), ('pinkard', 0.569), ('berry', 0.569)]
  NO stitch   

## 5. Anchor seeds and archetype building — both versions

In [9]:
# ── WITH STITCHING: original seeds ─────────────────────────────────────────────
ANCHOR_SEEDS_STITCH = {
    'score_god_given': [
        'explosive', 'burst', 'speed', 'quick_twitch', 'get_off',
        'twitch', 'twitchy', 'acceleration', 'first_step',
        'change_of_direction', 'agility', 'frame', 'size',
    ],
    'score_learned': [
        'technique', 'footwork', 'leverage', 'high_motor', 'motor',
        'effort', 'relentless', 'toughness', 'intelligence',
        'awareness', 'recognition', 'instinct', 'discipline',
        'pad_level', 'fundamental',
    ],
}

# ── WITHOUT STITCHING: translated to individual words ─────────────────────────
# quick_twitch → quick + twitch (both individually meaningful)
# get_off → explosive (already present) — drop since no token
# first_step → quickness (or keep as two words; W2V won't know "first_step")
# change_of_direction → agility (already present)
# high_motor → motor (already present)
# pad_level → leverage (semantic proxy)
ANCHOR_SEEDS_NO_STITCH = {
    'score_god_given': [
        'explosive', 'burst', 'speed', 'quick', 'twitch',
        'twitchy', 'acceleration', 'quickness', 'agility',
        'frame', 'size', 'athleticism', 'physical',
    ],
    'score_learned': [
        'technique', 'footwork', 'leverage', 'motor',
        'effort', 'relentless', 'toughness', 'intelligence',
        'awareness', 'recognition', 'instinct', 'discipline',
        'leverage', 'fundamental', 'coachable',
    ],
}
# dedupe
ANCHOR_SEEDS_NO_STITCH = {k: list(dict.fromkeys(v)) for k, v in ANCHOR_SEEDS_NO_STITCH.items()}

SCORE_COLS = list(ANCHOR_SEEDS_STITCH.keys())

# Vocab check
print('Seed vocab check:')
for pillar in SCORE_COLS:
    s_seeds = ANCHOR_SEEDS_STITCH[pillar]
    n_seeds = ANCHOR_SEEDS_NO_STITCH[pillar]
    in_s  = [s for s in s_seeds if s in w2v_s.wv]
    in_n  = [s for s in n_seeds if s in w2v_n.wv]
    miss_s = [s for s in s_seeds if s not in w2v_s.wv]
    miss_n = [s for s in n_seeds if s not in w2v_n.wv]
    print(f'  {pillar}')
    print(f'    STITCH   : {len(in_s)}/{len(s_seeds)} in W2V  MISSING={miss_s}')
    print(f'    NO-STITCH: {len(in_n)}/{len(n_seeds)} in W2V  MISSING={miss_n}')

Seed vocab check:
  score_god_given
    STITCH   : 13/13 in W2V  MISSING=[]
    NO-STITCH: 13/13 in W2V  MISSING=[]
  score_learned
    STITCH   : 15/15 in W2V  MISSING=[]
    NO-STITCH: 14/14 in W2V  MISSING=[]


In [10]:
def expand_pillar(seeds, model, topn=SEED_TOPN, threshold=SIM_THRESHOLD):
    candidate_scores = {}
    for seed in seeds:
        if seed not in model.wv:
            continue
        for word, sim in model.wv.most_similar(seed, topn=topn):
            if sim >= threshold and word not in seeds:
                candidate_scores.setdefault(word, []).append(sim)
    if not candidate_scores:
        return pd.DataFrame(columns=['term', 'seed_count', 'avg_sim', 'max_sim'])
    rows = [{'term': w, 'seed_count': len(s),
             'avg_sim': round(float(np.mean(s)), 3),
             'max_sim': round(float(np.max(s)),  3)}
            for w, s in candidate_scores.items()]
    return (pd.DataFrame(rows)
            .sort_values(['seed_count', 'avg_sim'], ascending=False)
            .reset_index(drop=True))


LEX_S = {p: expand_pillar(ANCHOR_SEEDS_STITCH[p],    w2v_s) for p in SCORE_COLS}
LEX_N = {p: expand_pillar(ANCHOR_SEEDS_NO_STITCH[p], w2v_n) for p in SCORE_COLS}

print('W2V expansion — high-confidence terms (seed_count >= 2):')
for pillar in SCORE_COLS:
    hc_s = LEX_S[pillar][LEX_S[pillar]['seed_count'] >= MIN_SEED_COUNT]
    hc_n = LEX_N[pillar][LEX_N[pillar]['seed_count'] >= MIN_SEED_COUNT]
    print(f'\n  {pillar}')
    print(f'    STITCH    ({len(hc_s)} terms): {hc_s["term"].head(15).tolist()}')
    print(f'    NO-STITCH ({len(hc_n)} terms): {hc_n["term"].head(15).tolist()}')

W2V expansion — high-confidence terms (seed_count >= 2):

  score_god_given
    STITCH    (38 terms): ['short_area', 'sheffield', 'top_end', 'reactive', 'quickness', 'athleticism', 'gilyard', 'chekwa', 'fluidity', 'gear', 'lafell', 'suddenness', 'sudden', 'explosiveness', 'twitched']
    NO-STITCH (44 terms): ['gilyard', 'sudden', 'sheffield', 'redmond', 'suddenness', 'gear', 'lafell', 'chekwa', 'length', 'explosiveness', 'nimble', 'standstill', 'powered', 'hardesty', 'overlapping']

  score_learned
    STITCH    (38 terms): ['anticipation', 'teamwork', 'rev', 'legree', 'pinkston', 'humming', 'lured', 'whistle', 'hardnosed', 'weatherspoon', 'bend', 'leadership', 'hustle', 'verner', 'persistence']
    NO-STITCH (30 terms): ['anticipation', 'iq', 'rev', 'pinkston', 'bored', 'whistle', 'cool', 'hustle', 'humming', 'duped', 'reading', 'teamwork', 'alem', 'zombie', 'energy']


In [11]:
MANUAL_STITCH = {
    'score_god_given': [
        'height', 'length', 'wingspan', 'natural', 'raw', 'powerful',
        'physical', 'vertical', 'leap', 'closing_speed', 'top_end',
        'body_control', 'fluidity', 'fluid', 'flexible', 'flexibility',
        'big', 'tall', 'wide', 'lean', 'fast', 'athlete',
        'rare', 'specimen', 'elite',
    ],
    'score_learned': [
        'pass_rush', 'pass_rusher', 'pass_protection', 'route_running',
        'block_shedding', 'anchor_strength', 'hand_fighting', 'hand_placement',
        'press_coverage', 'zone_coverage', 'run_blocking',
        'work_ethic', 'hard_worker', 'game_ready',
        'film_junkie', 'blue_collar', 'football_iq',
        'shed', 'disrupt', 'compete', 'hustle', 'grit',
        'smart', 'cerebral', 'coachable', 'diagnose', 'read', 'craft', 'savvy',
        'consistent', 'reliable', 'dependable', 'finish', 'skill',
    ],
}

# No-stitch manual: space-separated compounds → TF-IDF bigrams pick them up naturally
MANUAL_NO_STITCH = {
    'score_god_given': [
        'height', 'length', 'wingspan', 'natural', 'raw', 'powerful',
        'physical', 'vertical', 'leap', 'closing speed', 'top end',
        'body control', 'fluidity', 'fluid', 'flexible', 'flexibility',
        'big', 'tall', 'wide', 'lean', 'fast', 'athlete',
        'rare', 'specimen', 'elite',
    ],
    'score_learned': [
        'pass rush', 'pass rusher', 'pass protection', 'route running',
        'block shedding', 'anchor strength', 'hand fighting',
        'press coverage', 'zone coverage', 'run blocking',
        'work ethic', 'hard worker',
        'blue collar', 'football iq',
        'shed', 'disrupt', 'compete', 'hustle', 'grit',
        'smart', 'cerebral', 'coachable', 'diagnose', 'craft', 'savvy',
        'consistent', 'reliable', 'dependable', 'finish', 'skill',
    ],
}


def build_archetype_text(pillar, seeds, lexicon_df, manual):
    terms = list(seeds)
    hc = lexicon_df[lexicon_df['seed_count'] >= MIN_SEED_COUNT].head(MAX_W2V_TERMS)
    w2v_added = []
    for t in hc['term']:
        if t not in terms and t.replace('_', '').isalpha() and len(t) > 2:
            terms.append(t)
            w2v_added.append(t)
    for t in manual:
        if t not in terms:
            terms.append(t)
    return ' '.join(terms), w2v_added


ARCH_S, ARCH_N = {}, {}
W2V_CONTRIB_S, W2V_CONTRIB_N = {}, {}

for pillar in SCORE_COLS:
    ARCH_S[pillar], W2V_CONTRIB_S[pillar] = build_archetype_text(
        pillar, ANCHOR_SEEDS_STITCH[pillar], LEX_S[pillar], MANUAL_STITCH[pillar])
    ARCH_N[pillar], W2V_CONTRIB_N[pillar] = build_archetype_text(
        pillar, ANCHOR_SEEDS_NO_STITCH[pillar], LEX_N[pillar], MANUAL_NO_STITCH[pillar])

print('Archetype sizes:')
for pillar in SCORE_COLS:
    n_s = len(ARCH_S[pillar].split())
    n_n = len(ARCH_N[pillar].split())
    print(f'  {pillar}: STITCH={n_s} tokens  NO-STITCH={n_n} tokens')

Archetype sizes:
  score_god_given: STITCH=61 tokens  NO-STITCH=64 tokens
  score_learned: STITCH=73 tokens  NO-STITCH=80 tokens


## 6. TF-IDF vectorization — separate vectorizers

In [12]:
def fit_vectorizer(df, cols, token_pattern, label):
    all_text = pd.concat([df[c] for c in cols], ignore_index=True)
    vec = TfidfVectorizer(
        ngram_range    = (1, 2),
        min_df         = 2,
        max_df         = 0.95,
        sublinear_tf   = True,
        token_pattern  = token_pattern,
    )
    vec.fit(all_text)
    print(f'[{label}] Vocabulary: {len(vec.vocabulary_):,} features')
    return vec


vec_s = fit_vectorizer(df, ['str_s', 'ov_s', 'wk_s'], r'[a-z_]+', 'WITH stitch')
vec_n = fit_vectorizer(df, ['str_n', 'ov_n', 'wk_n'], r'[a-z]+',  'NO stitch')

# Transform player matrices
mat_str_s  = vec_s.transform(df['str_s'])
mat_wk_s   = vec_s.transform(df['wk_s'])
mat_ov_s   = vec_s.transform(df['ov_s'])

mat_str_n  = vec_n.transform(df['str_n'])
mat_wk_n   = vec_n.transform(df['wk_n'])
mat_ov_n   = vec_n.transform(df['ov_n'])

# ── Key diagnostic: IDF balance of stitched vs individual tokens ──────────────
print('\nIDF comparison — stitched token vs constituent words:')
idf_s = dict(zip(vec_s.get_feature_names_out(), vec_s.idf_))
idf_n = dict(zip(vec_n.get_feature_names_out(), vec_n.idf_))

compound_checks = [
    ('pass_rush',           'pass',     'rush'),
    ('high_motor',          'high',     'motor'),
    ('quick_twitch',        'quick',    'twitch'),
    ('route_running',       'route',    'running'),
    ('change_of_direction', 'change',   'direction'),
    ('pad_level',           'pad',      'level'),
    ('body_control',        'body',     'control'),
    ('anchor_strength',     'anchor',   'strength'),
]

rows = []
for stitch_tok, w1, w2 in compound_checks:
    bigram_tok = f'{w1} {w2}'   # how TF-IDF bigram looks in no-stitch vocab
    rows.append({
        'term'          : stitch_tok,
        'IDF_stitched'  : round(idf_s.get(stitch_tok, np.nan), 3),
        'IDF_w1'        : round(idf_n.get(w1, np.nan), 3),
        'IDF_w2'        : round(idf_n.get(w2, np.nan), 3),
        'IDF_bigram'    : round(idf_n.get(bigram_tok, np.nan), 3),
    })

idf_cmp = pd.DataFrame(rows)
print(idf_cmp.to_string(index=False))
print('\nHigher IDF = rarer = more discriminative weight in cosine similarity.')
print('If IDF_stitched >> IDF_bigram, stitching is inflating the weight of compound terms.')

[WITH stitch] Vocabulary: 106,670 features
[NO stitch] Vocabulary: 105,698 features

IDF comparison — stitched token vs constituent words:
               term  IDF_stitched  IDF_w1  IDF_w2  IDF_bigram
          pass_rush         4.623   4.547   2.997       9.808
         high_motor         6.918   3.171   4.551       6.918
       quick_twitch         6.385   3.279   4.684       6.312
      route_running         5.206   2.478   3.290       5.193
change_of_direction         4.363   3.855   3.892       4.112
          pad_level         4.223   3.646   5.157         NaN
       body_control         4.004   2.785   3.512       4.003
    anchor_strength         7.506   4.069   2.628       7.473

Higher IDF = rarer = more discriminative weight in cosine similarity.
If IDF_stitched >> IDF_bigram, stitching is inflating the weight of compound terms.


In [13]:
# Transform archetype texts through their respective vectorizers
arch_texts_s = [preprocess_stitch(ARCH_S[col], extra_phrases=AUTO_PHRASE_MAP) for col in SCORE_COLS]
arch_texts_n = [preprocess_no_stitch(ARCH_N[col]) for col in SCORE_COLS]

arch_mat_s = vec_s.transform(arch_texts_s)
arch_mat_n = vec_n.transform(arch_texts_n)

print('Archetype vector density (non-zero features = vocabulary match):')
print(f'{"Pillar":20s}  {"STITCH nnz":>12s}  {"NO-STITCH nnz":>14s}')
for i, col in enumerate(SCORE_COLS):
    print(f'  {col:20s}  {arch_mat_s[i].nnz:12d}  {arch_mat_n[i].nnz:14d}')

print('\nHigher nnz = more corpus terms matched = richer archetype coverage.')

Archetype vector density (non-zero features = vocabulary match):
Pillar                  STITCH nnz   NO-STITCH nnz
  score_god_given                 77              86
  score_learned                   80             107

Higher nnz = more corpus terms matched = richer archetype coverage.


## 7. Score both pipelines and compare

In [14]:
def score_pipeline(mat_str, mat_wk, mat_ov, arch_mat):
    sim_str = cosine_similarity(mat_str, arch_mat)
    sim_wk  = cosine_similarity(mat_wk,  arch_mat)
    sim_ov  = cosine_similarity(mat_ov,  arch_mat)
    raw = W_STRENGTHS * sim_str + W_WEAKNESSES * sim_wk + W_OVERVIEW * sim_ov
    raw_df = pd.DataFrame(raw, columns=SCORE_COLS)
    scaler = MinMaxScaler(feature_range=(0, 100))
    scaled = pd.DataFrame(scaler.fit_transform(raw_df), columns=SCORE_COLS).round(1)
    return raw_df, scaled, sim_str, sim_wk


raw_s, scores_s, sim_str_s, sim_wk_s = score_pipeline(mat_str_s, mat_wk_s, mat_ov_s, arch_mat_s)
raw_n, scores_n, sim_str_n, sim_wk_n = score_pipeline(mat_str_n, mat_wk_n, mat_ov_n, arch_mat_n)

print('Score distribution comparison (0–100 scaled):')
print(f'{"":30s}  {"WITH STITCH":>30s}  {"NO STITCH":>30s}')
print(f'{"Metric":30s}  {"god_given":>14s} {"learned":>14s}  {"god_given":>14s} {"learned":>14s}')
print('-' * 96)
for stat in ['mean', 'std', 'min', '25%', '50%', '75%', 'max']:
    d_s = scores_s[SCORE_COLS].describe().loc[stat]
    d_n = scores_n[SCORE_COLS].describe().loc[stat]
    print(f'  {stat:28s}  {d_s[SCORE_COLS[0]]:14.2f} {d_s[SCORE_COLS[1]]:14.2f}  '
          f'{d_n[SCORE_COLS[0]]:14.2f} {d_n[SCORE_COLS[1]]:14.2f}')

Score distribution comparison (0–100 scaled):
                                                   WITH STITCH                       NO STITCH
Metric                               god_given        learned       god_given        learned
------------------------------------------------------------------------------------------------
  mean                                   47.89          49.63           53.32          45.51
  std                                     8.45           9.36            9.08           7.54
  min                                     0.00           0.00            0.00           0.00
  25%                                    43.00          44.00           48.08          41.10
  50%                                    47.60          48.20           52.90          45.20
  75%                                    52.60          54.80           58.40          49.70
  max                                   100.00         100.00          100.00         100.00


In [15]:
# ── Rank correlation: are the two versions ranking players similarly? ──────────
print('RANK CORRELATION (Spearman ρ) — WITH vs WITHOUT stitching')
print('ρ close to 1.0 → consistent rankings; ρ < 0.8 → meaningful divergence')
print()
for col in SCORE_COLS:
    rho, pval = spearmanr(scores_s[col], scores_n[col])
    print(f'  {col:25s}  ρ = {rho:.4f}  (p={pval:.2e})')

# ── Variance inflation: does stitching inflate variance? ─────────────────────
print('\nRAW SCORE VARIANCE (before min-max scaling):')
print(f'{"Pillar":25s}  {"Var WITH stitch":>18s}  {"Var NO stitch":>16s}  {"Ratio S/N":>10s}')
for col in SCORE_COLS:
    v_s = raw_s[col].var()
    v_n = raw_n[col].var()
    ratio = v_s / v_n if v_n > 0 else float('inf')
    print(f'  {col:25s}  {v_s:18.6f}  {v_n:16.6f}  {ratio:10.3f}')
print('\nRatio > 1 → stitching increases variance (potentially inflated separation).')
print('Ratio < 1 → no-stitch spreads scores more widely.')

RANK CORRELATION (Spearman ρ) — WITH vs WITHOUT stitching
ρ close to 1.0 → consistent rankings; ρ < 0.8 → meaningful divergence

  score_god_given            ρ = 0.8737  (p=0.00e+00)
  score_learned              ρ = 0.7593  (p=0.00e+00)

RAW SCORE VARIANCE (before min-max scaling):
Pillar                        Var WITH stitch     Var NO stitch   Ratio S/N
  score_god_given                      0.000253          0.000206       1.229
  score_learned                        0.000130          0.000192       0.675

Ratio > 1 → stitching increases variance (potentially inflated separation).
Ratio < 1 → no-stitch spreads scores more widely.


In [16]:
# ── Top-10 agreement per pillar ───────────────────────────────────────────────
base_cols = ['player_name', 'position', 'Pos_Group']
result_s = pd.concat([df[base_cols].reset_index(drop=True), scores_s], axis=1)
result_n = pd.concat([df[base_cols].reset_index(drop=True), scores_n], axis=1)

print('TOP-10 AGREEMENT — does stitching vs no-stitching rank the same players?')
print('=' * 80)
for col in SCORE_COLS:
    top_s = set(result_s.nlargest(10, col)['player_name'])
    top_n = set(result_n.nlargest(10, col)['player_name'])
    overlap = top_s & top_n
    only_s  = top_s - top_n
    only_n  = top_n - top_s
    print(f'\n{col.replace("score_","").upper()} — {len(overlap)}/10 shared')
    print(f'  Both agree on : {sorted(overlap)}')
    print(f'  Only in STITCH: {sorted(only_s)}')
    print(f'  Only in NO-STI: {sorted(only_n)}')

# ── Positional archetypes ─────────────────────────────────────────────────────
print('\n\nPOSITION GROUP MEDIAN SCORES (0–100 scale):')
print(f'{"":20s}  {"--- WITH STITCH ---":^30s}  {"--- NO STITCH ---":^30s}')
header = f'{"Pos_Group":20s}  {"god_given":>14s} {"learned":>14s}  {"god_given":>14s} {"learned":>14s}'
print(header)
print('-' * len(header))

med_s = result_s.groupby('Pos_Group')[SCORE_COLS].median().round(1)
med_n = result_n.groupby('Pos_Group')[SCORE_COLS].median().round(1)
all_pos = sorted(set(med_s.index) | set(med_n.index))
for pos in all_pos:
    rs = med_s.loc[pos] if pos in med_s.index else pd.Series([np.nan]*len(SCORE_COLS), index=SCORE_COLS)
    rn = med_n.loc[pos] if pos in med_n.index else pd.Series([np.nan]*len(SCORE_COLS), index=SCORE_COLS)
    print(f'  {pos:20s}  {rs[SCORE_COLS[0]]:14.1f} {rs[SCORE_COLS[1]]:14.1f}  '
          f'{rn[SCORE_COLS[0]]:14.1f} {rn[SCORE_COLS[1]]:14.1f}')

TOP-10 AGREEMENT — does stitching vs no-stitching rank the same players?

GOD_GIVEN — 6/10 shared
  Both agree on : ['Alex Parsons', 'Chris Westry', 'Daniel Hardy', 'DeMarcus Lawrence', 'Malcolm Koonce', 'Oluwole Betiku Jr.']
  Only in STITCH: ['Chase Young', 'Haason Reddick', 'Jordon Riley', 'Menelik Watson']
  Only in NO-STI: ['Jason Fox', 'Jaylon McClain-Sapp', 'Kevin Thomas', 'Taylor Mays']

LEARNED — 3/10 shared
  Both agree on : ['Austin Wentworth', 'Chris Watt', 'Rahim Alem']
  Only in STITCH: ['Arthur Moats', 'Cam Jones', 'Derrick Morgan', 'Mika Tafua', 'Stone Blanton', 'Victor Johnson', 'Will Clarke']
  Only in NO-STI: ['Brandon Fusco', 'Brian Rolle', 'Buster Skrine', 'Donald Hawkins', 'James Robinson', 'Warren McClendon Jr.', 'Will Clapp']


POSITION GROUP MEDIAN SCORES (0–100 scale):
                           --- WITH STITCH ---              --- NO STITCH ---       
Pos_Group                  god_given        learned       god_given        learned
--------------------------

In [17]:
# ── Score imbalance: are scores clustered or spread? ──────────────────────────
# For 2-bin, we want clear separation between pillars per player
# (god_given ≠ learned for most players — otherwise the 2-bin is meaningless)

# Compute per-player pillar gap |god_given - learned|
gap_s = (scores_s[SCORE_COLS[0]] - scores_s[SCORE_COLS[1]]).abs()
gap_n = (scores_n[SCORE_COLS[0]] - scores_n[SCORE_COLS[1]]).abs()

print('PILLAR SEPARATION (|god_given - learned| per player):')
print(f'{"":15s}  {"WITH STITCH":>14s}  {"NO STITCH":>12s}')
for stat_name, s_val, n_val in [
    ('Mean gap',   gap_s.mean(),   gap_n.mean()),
    ('Median gap', gap_s.median(), gap_n.median()),
    ('Std gap',    gap_s.std(),    gap_n.std()),
    ('% gap < 5',  (gap_s < 5).mean()*100, (gap_n < 5).mean()*100),
    ('% gap > 20', (gap_s > 20).mean()*100, (gap_n > 20).mean()*100),
]:
    print(f'  {stat_name:15s}  {s_val:14.2f}  {n_val:12.2f}')

print('\n% gap < 5 → players where both pillars score nearly identically (poor discrimination).')
print('% gap > 20 → players with clear pillar dominance (good discrimination).')

# ── Correlation between pillars (should be low for 2-bin to be meaningful) ────
print('\nCORRELATION BETWEEN PILLARS (Pearson):')
corr_s = scores_s[SCORE_COLS].corr().iloc[0, 1]
corr_n = scores_n[SCORE_COLS].corr().iloc[0, 1]
print(f'  WITH stitch : {corr_s:.4f}')
print(f'  NO stitch   : {corr_n:.4f}')
print('Lower correlation = pillars are more orthogonal = better 2-bin taxonomy.')

PILLAR SEPARATION (|god_given - learned| per player):
                    WITH STITCH     NO STITCH
  Mean gap                  10.61         12.03
  Median gap                 8.40         10.10
  Std gap                    8.95          9.29
  % gap < 5                 31.58         25.16
  % gap > 20                13.65         17.37

% gap < 5 → players where both pillars score nearly identically (poor discrimination).
% gap > 20 → players with clear pillar dominance (good discrimination).

CORRELATION BETWEEN PILLARS (Pearson):
  WITH stitch : -0.1937
  NO stitch   : -0.2245
Lower correlation = pillars are more orthogonal = better 2-bin taxonomy.


In [18]:
# ── Diagnostic summary ────────────────────────────────────────────────────────
print('=' * 70)
print('ABLATION SUMMARY')
print('=' * 70)

# 1. Vocab size
print(f'\n1. VOCABULARY')
print(f'   WITH stitch  : {len(vec_s.vocabulary_):,} TF-IDF features  |  {len([t for t in vocab_s if "_" in t])} stitched tokens')
print(f'   NO stitch    : {len(vec_n.vocabulary_):,} TF-IDF features  |  0 stitched tokens')

# 2. IDF inflation
avg_idf_stitch_toks = np.mean([idf_s[t] for t in idf_s if '_' in t])
avg_idf_plain_toks  = np.mean([idf_s[t] for t in idf_s if '_' not in t])
print(f'\n2. IDF INFLATION')
print(f'   Avg IDF of stitched tokens (WITH): {avg_idf_stitch_toks:.3f}')
print(f'   Avg IDF of plain tokens    (WITH): {avg_idf_plain_toks:.3f}')
print(f'   Ratio: {avg_idf_stitch_toks/avg_idf_plain_toks:.2f}x  (>1.0 confirms stitched tokens have inflated weights)')

# 3. Archetype density
print(f'\n3. ARCHETYPE VECTOR DENSITY (non-zero features)')
for i, col in enumerate(SCORE_COLS):
    print(f'   {col}: STITCH={arch_mat_s[i].nnz}  NO-STITCH={arch_mat_n[i].nnz}')

# 4. Rank correlation
print(f'\n4. RANK CORRELATION (Spearman ρ between versions)')
for col in SCORE_COLS:
    rho, _ = spearmanr(scores_s[col], scores_n[col])
    print(f'   {col}: ρ = {rho:.4f}')

# 5. Pillar discrimination
print(f'\n5. PILLAR DISCRIMINATION')
print(f'   WITH stitch  — mean |gap|: {gap_s.mean():.2f}  pillar corr: {corr_s:.4f}')
print(f'   NO stitch    — mean |gap|: {gap_n.mean():.2f}  pillar corr: {corr_n:.4f}')

# 6. Recommendation
print(f'\n6. RECOMMENDATION')
if avg_idf_stitch_toks / avg_idf_plain_toks > 1.3:
    print('   ⚠ Stitched tokens have >1.3x higher avg IDF than plain tokens.')
    print('   This confirms stitching is inflating compound-term weights in cosine similarity.')
if corr_n < corr_s:
    print('   ✓ No-stitch produces more orthogonal pillars (lower inter-pillar correlation).')
else:
    print('   ✗ Stitching produces more orthogonal pillars — stitching may be helping.')
if gap_n.mean() > gap_s.mean():
    print('   ✓ No-stitch produces larger per-player pillar gaps (better player discrimination).')
else:
    print('   ✗ Stitching produces larger pillar gaps — may be providing useful signal.')

ABLATION SUMMARY

1. VOCABULARY
   WITH stitch  : 106,670 TF-IDF features  |  120 stitched tokens
   NO stitch    : 105,698 TF-IDF features  |  0 stitched tokens

2. IDF INFLATION
   Avg IDF of stitched tokens (WITH): 9.348
   Avg IDF of plain tokens    (WITH): 9.239
   Ratio: 1.01x  (>1.0 confirms stitched tokens have inflated weights)

3. ARCHETYPE VECTOR DENSITY (non-zero features)
   score_god_given: STITCH=77  NO-STITCH=86
   score_learned: STITCH=80  NO-STITCH=107

4. RANK CORRELATION (Spearman ρ between versions)
   score_god_given: ρ = 0.8737
   score_learned: ρ = 0.7593

5. PILLAR DISCRIMINATION
   WITH stitch  — mean |gap|: 10.61  pillar corr: -0.1937
   NO stitch    — mean |gap|: 12.03  pillar corr: -0.2245

6. RECOMMENDATION
   ✓ No-stitch produces more orthogonal pillars (lower inter-pillar correlation).
   ✓ No-stitch produces larger per-player pillar gaps (better player discrimination).
